# StockSense Pipeline Runner
**Run this notebook in Google Colab to populate the Neon database.**

----

## Before you run anything, read this doc first

### 1. Enable GPU (required for FinBERT in Cell 8)
- Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU** -> Save
- Do this BEFORE running any cells
- If you already started without GPU: Runtime -> Restart session, then re-run from the top

### 2. Set up Colab Secrets (required, do this once)
Click the key icon in the left sidebar, then add:

| Secret Name   | What to put there                                      | Required for |
|---------------|--------------------------------------------------------|--------------|
| `sshkey`     | Your GitHub SSH private key (`~/.ssh/id_ed25519`)      | All cells    |
| `DATABASE_URL`| The Neon PostgreSQL connection string (get from Katie) | All cells    |
| `HUGGINGFACE_HUB_TOKEN` | Kevin's HuggingFace API token (from hf.co/settings/tokens) | Cell 7 |
| `MARKETAUX_API_TOKEN` | Marketaux API token (for recent news) | Cell 11 |
| `OPENAI_API_KEY` | OpenAI API key (for GPT summaries) | Cell 12 |

Toggle Notebook access ON for all secrets you plan to use.

Ask Katie for the DATABASE_URL if you don't have it.

### 3. Run order
| Cell | What it does | Required? |
|------|-------------|-----------|
| 1 | Clone repo | Yes |
| 2 | Navigate to backend | Yes |
| 3 | Install dependencies | Yes |
| 4 | Load secrets into env | Yes |
| 5 | Create DB tables | Yes (first time only) |
| 6 | Ingest stock price data | Yes |
| 7 | Ingest historical news articles (filtered HF subsets) | Yes |
| 8 | Run FinBERT sentiment scoring | Yes (GPU required) |
| 9 | Calculate stock returns | Yes |
| 10 | Aggregate sentiment snapshots | Yes |
| 11 | Ingest recent news via Marketaux | Optional |
| 12 | Generate GPT news summaries | Optional |

### 4. Never commit our secrets
Before saving this notebook to GitHub:
**Edit -> Clear all outputs** -> this removes any printed values from cells

In [ ]:
# ============================================================
# CELL 1: Clone private repo using SSH key from Colab Secrets
# ============================================================
import os
from google.colab import userdata

# Write private key to disk (fix Windows line endings)
os.makedirs("/root/.ssh", exist_ok=True)
key = userdata.get("sshkey")
key = key.replace("\r\n", "\n").replace("\r", "\n")
if not key.endswith("\n"):
    key += "\n"

with open("/root/.ssh/id_ed25519", "w") as f:
    f.write(key)
os.chmod("/root/.ssh/id_ed25519", 0o600)

# Add GitHub to known hosts (prevents interactive prompt)
!ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null

# Clone the repo
!git clone git@github.com:SCCapstone/ai_roosters.git

print("Clone complete.")


In [ ]:
# ============================================================
# CELL 2: Navigate into the backend folder
# ============================================================
%cd /content/ai_roosters/backend

# Verify in the right place
!pwd
!ls  # should show: app/, requirements-pipeline.txt, API.Dockerfile, etc.

In [ ]:
# Cell 3 - install system build tools first, then packages
!apt-get install -y --quiet libpq-dev gcc build-essential
# Install core packages (known-good binary wheels)
!pip install --prefer-binary \
    fastapi==0.95.2 uvicorn==0.22.0 sqlalchemy==1.4.41 \
    psycopg2-binary==2.9.6 "pydantic[email]==1.10.9" \
    python-jose[cryptography] "passlib[bcrypt]==1.7.4" bcrypt==4.0.1 \
    pandas numpy yfinance requests httpx python-dotenv \
    email-validator openai \
    torch transformers xgboost scikit-learn datasets matplotlib

# Skip papermill install, it causes errors.
# Papermill is not needed for running the pipeline scripts

In [ ]:
# ============================================================
# CELL 4: Load secrets into environment variables
# DATABASE_URL is the Neon connection string - stored in
# Colab Secrets (ask Katie for URL)
# ============================================================
from google.colab import userdata
import os, sys

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
os.environ["HUGGINGFACE_HUB_TOKEN"] = userdata.get("HUGGINGFACE_HUB_TOKEN")
os.environ["HF_TOKEN"] = os.environ["HUGGINGFACE_HUB_TOKEN"]

# Optional secrets — only needed for cells 11 and 12
try:
    os.environ["MARKETAUX_API_TOKEN"] = userdata.get("MARKETAUX_API_TOKEN")
except Exception:
    pass  # skip if not configured

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # skip if not configured

# add backend to Python path so imports work
sys.path.insert(0, '/content/ai_roosters/backend')

print("Environment ready.")
print(f"DATABASE_URL set:            {'yes' if os.environ.get('DATABASE_URL') else 'NO - check the secret name'}")
print(f"HUGGINGFACE_HUB_TOKEN set:   {'yes' if os.environ.get('HUGGINGFACE_HUB_TOKEN') else 'NO - check the secret name'}")
print(f"MARKETAUX_API_TOKEN set:     {'yes' if os.environ.get('MARKETAUX_API_TOKEN') else 'not set (needed for cell 11)'}")
print(f"OPENAI_API_KEY set:          {'yes' if os.environ.get('OPENAI_API_KEY') else 'not set (needed for cell 12)'}")

In [ ]:
# ============================================================
# CELL 5: Create all tables in Neon (safe to re-run ,
# uses CREATE TABLE IF NOT EXISTS )
# ============================================================

from app.db_init import init_db

init_db()
print("Database tables initialized.")



In [ ]:
# ============================================================
# CELL 6: Download price data from yfinance
# and store it in the Neon 'stocks' table.
#
# update_existing=False means: skip rows that already exist.
# Safe to re-run, won't create duplicates.
# ============================================================
from app.services.ingesting_pipelines.prices_ingest import PriceIngestor
from datetime import date

ingestor = PriceIngestor()

tickers = [
    "KSS",   # Kohl's
    "ALK",   # Alaska Air
    "NVS",   # Novartis
    "AXP",   # American Express
    "FCX",   # Freeport-McMoRan
    "CSX",   # CSX Corporation
    "DAL",   # Delta Air Lines
    "NTAP",  # NetApp
    "AMZN",  # Amazon
    "AAPL",  # Apple
    "MRK",   # Merck
    "NVDA",  # Nvidia
    "COP",   # ConocoPhillips
    "BHP",   # BHP Group
    "EA",    # Electronic Arts
]

ingestor.ingest_multiple_stocks(
    tickers=tickers,
    start_date="2020-01-01",
    end_date=str(date.today()),
    period=None,
    update_existing=False,
)

print("Price ingestion complete.")

In [ ]:
# ============================================================
# CELL 7: Ingest historical news articles (filtered subsets)
# Imports ArticleIngestor from the cloned backend, then adds
# named HuggingFace subset support via a small subclass.
# ============================================================
import os
from datetime import date
from typing import Optional
from datasets import load_dataset
from app.services.ingesting_pipelines.news_ingest import ArticleIngestor as _Base


class ArticleIngestor(_Base):
    """Adds named parquet-subset loading to the backend ArticleIngestor."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._active_name: Optional[str] = None

    def load_dataset_any(self, streaming: bool):
        token = (os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN") or "").strip() or None
        if self._active_name:
            files = f"hf://datasets/Brianferrell787/financial-news-multisource/data/{self._active_name}/*.parquet"
            return load_dataset("parquet", data_files=files, split="train", streaming=streaming, token=token)
        return super().load_dataset_any(streaming)

    def ingest_all_years_one_pass(self, *args, name: Optional[str] = None, **kwargs):
        self._active_name = name
        try:
            return super().ingest_all_years_one_pass(*args, **kwargs)
        finally:
            self._active_name = None


ing = ArticleIngestor()
YEARS = list(range(2020, date.today().year + 1))
END   = str(date.today())
OPTS  = dict(years=YEARS, per_year=1000, end_date=END, streaming=False)

ing.ingest_all_years_one_pass(**OPTS, name="yahoo_finance_felixdrinkall")
ing.ingest_all_years_one_pass(**OPTS, name="benzinga_6000stocks")
ing.ingest_all_years_one_pass(**OPTS, name="reddit_finance_sp500")

print("News ingest complete.")

In [ ]:
# ============================================================
# CELL 8: Run FinBERT sentiment scoring on ingested articles
# Scores unprocessed rows in 'articles' and
# 'stock_news_articles'. GPU (T4) required for speed.
# ============================================================
import os
os.environ["FINBERT_TABLES"] = "articles,stock_news_articles"
os.environ["FINBERT_ONLY_MISSING"] = "true"
os.environ["FINBERT_TOTAL"] = "10000"
os.environ["FINBERT_FETCH_BATCH"] = "500"

from app.services.sentiment.article_processing import run_finbert_pipeline_from_env

result = run_finbert_pipeline_from_env()
print("FinBERT processing complete.")

In [ ]:
# ============================================================
# CELL 9: Calculate stock returns from price data
# Computes daily/weekly/monthly return columns used by
# the sentiment correlation models.
# ============================================================
from app.services.sentiment.stock_processing import run_returns_pipeline

run_returns_pipeline()
print("Returns pipeline complete.")

In [ ]:
# ============================================================
# CELL 10: Aggregate sentiment snapshots per ticker
# Combines price + sentiment data into snapshot rows used
# by the frontend charts.
# ============================================================
import os
from datetime import date

os.environ["AGG_TICKERS"] = "KSS,ALK,NVS,AXP,FCX,CSX,DAL,NTAP,AMZN,AAPL,MRK,NVDA,COP,BHP,EA"
os.environ["AGG_START_DATE"] = "2020-01-01"
os.environ["AGG_END_DATE"] = str(date.today())

from app.services.sentiment.aggregator import run_sentiment_snapshot_pipeline_from_env

result = run_sentiment_snapshot_pipeline_from_env()
print("Aggregator complete.")

In [ ]:
# ============================================================
# CELL 11: Ingest recent news via Marketaux API
# Fetches the latest stock news for all target tickers.
# Requires MARKETAUX_API_TOKEN secret.
# ============================================================
from app.services.ingesting_pipelines.daily_news_ingest import StockNewsIngestor

ing = StockNewsIngestor()
ing.ingest_recent_news()

print("Daily news ingest complete.")

In [ ]:
# ============================================================
# CELL 12: Generate GPT news summaries
# Summarizes recent stock news for 7-day and 30-day windows.
# Requires OPENAI_API_KEY secret.
# ============================================================
import os
os.environ["NEWS_SUMMARY_WINDOWS"] = "7,30"

from app.services.sentiment.gpt_summary import StockNewsSummaryGenerator

gen = StockNewsSummaryGenerator()
gen.run(
    tickers=None,                  # uses all target tickers
    windows=None,                  # uses NEWS_SUMMARY_WINDOWS env var
    max_articles_per_summary=8,
)

print("GPT summaries complete.")